# Profitable Wallet Analysis

Loads processed trades from `data/trades_processed/` (top-5% wallets by P&L)

In [ ]:
import json
import math
from pathlib import Path

import pandas as pd
import numpy as np

## Parameters

In [ ]:
TRADES_PROCESSED = Path("../data/trades_processed")

## 1 · Load processed trades

In [ ]:
TRADES_PARQUET = Path("../data/trades_processed.parquet")

if TRADES_PARQUET.exists():
    # Fast path: load from parquet
    df = pd.read_parquet(TRADES_PARQUET)
else:
    # Slow path: load from jsonl files
    rows = []
    for jsonl_path in sorted(TRADES_PROCESSED.rglob("*.jsonl")):
        with jsonl_path.open() as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df["dt"]               = pd.to_datetime(df["dt"], utc=True)
    df["trade_value_usdc"] = df["trade_value_usdc"].astype(float)
    df["final_value_usdc"] = df["final_value_usdc"].astype(float)
    df["size"]             = df["size"].astype(float)
    df["price"]            = df["price"].astype(float)
    
    # Save for next time
    df.to_parquet(TRADES_PARQUET, index=False)

# calculate pnl
df["pnl"] = np.where(
    df["side"] == "BUY", 
    df["final_value_usdc"] - df["trade_value_usdc"], 
    # we buy the opposite token for (1-p) * size = size - trade_value
    #  final val is (1-final_price) * size = size - final_price * size
    # so pnl is (size - final_price * size) - (size - trade_value) = trade_value - final_price * size
    df["trade_value_usdc"] - df["final_value_usdc"]
)

print(f"Loaded {len(df):,} trade records for {df['wallet'].nunique():,} wallets.")
df.head(3)

In [ ]:
df[df['side'] == "SELL"].head(3)

In [ ]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
df.groupby("wallet").agg(
    num_trades = ("transactionHash", "count"),
    num_markets = ("condition_id", "nunique"),
    pnl_usdc    = ("pnl", "sum")
).describe(percentiles=QUANTILES)



## 2 · Weighted-pnl volatility per wallet

Step 1 — collapse fills into **timestamp buckets** `(wallet, dt)`.  
Step 2 — apply the weighted-return volatility formula across those buckets.

In [ ]:
import math
import numpy as np
import pandas as pd

def scaled_weighted_pnl_volatility(buckets: pd.DataFrame) -> float:
    """
    Compute capital-weighted PnL volatility scaled by sqrt(total PnL).

    Each row of `buckets` must contain:
        - notional : total capital in the bucket
        - pnl      : realized PnL in the bucket
    """
    if len(buckets) < 2:
        return float("nan")

    w = buckets["notional"].to_numpy()
    pnl = buckets["pnl"].to_numpy()

    total_w = w.sum()
    total_pnl = pnl.sum()

    if total_w == 0 or total_pnl <= 0:
        return float("nan")

    # weighted mean PnL
    mean = np.sum(w * pnl) / total_w

    # weighted variance
    variance = np.sum(w * (pnl - mean) ** 2) / total_w
    sigma = math.sqrt(variance)

    # scale by sqrt of total PnL
    sigma_scaled = sigma / math.sqrt(total_pnl)

    return sigma_scaled

In [ ]:
# ── Step 1: aggregate fills → timestamp buckets per wallet ───────────────────
df["notional"] = df.apply(
    lambda r: r["trade_value_usdc"] if r["side"] == "BUY" else r['size'] * (1-r["price"]), 
    axis=1
)

# floor to the nearest 5 minutes
df["dt_floored"] = df["dt"].dt.floor("5T") 

buckets = (
    df.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
    .agg(
        notional=("notional", "sum"),
        pnl=("pnl", "sum"),
    )
    .reset_index()
)

# drop buckets with zero notional (e.g. dust / zero-cost trades)
buckets = buckets[buckets["notional"] > 0].copy()

print(f"Timestamp buckets: {len(buckets):,}  across {buckets['wallet'].nunique():,} wallets.")
buckets.head(5)

In [ ]:
WALLET = "0xa49becb692927d455924583b5e3e5788246f4c40"

print("PnL: " + str(df[df['wallet'] == WALLET]['pnl'].sum()))

(buckets[buckets["wallet"] == WALLET]
 .groupby("wallet")
 .apply(scaled_weighted_pnl_volatility, include_groups=False)
 .head(5))

In [ ]:
# ── Step 2: volatility per wallet ────────────────────────────────────────────

def wallet_metrics(group: pd.DataFrame) -> pd.Series:
    """Compute all wallet metrics in one pass."""
    pnl = group["pnl"].to_numpy()
    total_notional = group["notional"].sum()
    total_pnl = pnl.sum()
    
    # top-5 pnl concentration
    if total_pnl <= 0:
        top5_pnl_pct = float("nan")
        top_market_pnl_pct = float("nan")
    else:
        top5_pnl = np.sort(pnl)[-5:].sum()
        top5_pnl_pct = top5_pnl / total_pnl
        top_market_pnl_pct = group.groupby("condition_id")["pnl"].sum().max() / total_pnl
    
    return pd.Series({
        "pnl_volatility": scaled_weighted_pnl_volatility(group),
        "num_buckets": len(group),
        "num_markets": len(group['condition_id'].unique()),
        "total_notional": total_notional,
        "total_pnl": total_pnl,
        "top5_pnl_pct": top5_pnl_pct,
        "top_market_pnl_pct": top_market_pnl_pct,
    })

wallet_vol = (
    buckets.groupby("wallet", sort=False)
    .apply(wallet_metrics, include_groups=False)
    .reset_index()
)

wallet_vol["return"] = wallet_vol["total_pnl"] / wallet_vol["total_notional"]
wallet_vol = wallet_vol.sort_values("pnl_volatility").reset_index(drop=True)

print(f"Wallets with volatility: {wallet_vol['pnl_volatility'].notna().sum():,}")
wallet_vol.head(10)

In [ ]:
wallet_vol[wallet_vol["wallet"] == WALLET]

In [ ]:
# df[df["wallet"] == WALLET][['dt', 'title', 'side', 'size', 'price', 'pnl']].sort_values('dt')

## 3 · Volatility distribution

In [ ]:
QUANTILES = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
N = wallet_vol["pnl_volatility"].notna().sum()

vol_stats = (
    wallet_vol["pnl_volatility"]
    .dropna()
    .quantile(QUANTILES)
    .rename_axis("quantile")
    .to_frame()
)
vol_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
vol_stats.loc["mean"] = [float("nan"), wallet_vol["pnl_volatility"].mean()]
vol_stats.loc["std"]  = [float("nan"), wallet_vol["pnl_volatility"].std()]
vol_stats["wallet_count"] = vol_stats["wallet_count"].astype("Int64")
vol_stats["pnl_volatility"] = vol_stats["pnl_volatility"].round(4)

vol_stats

In [ ]:
len(wallet_vol)

In [ ]:
len(wallet_vol[(wallet_vol['num_buckets'] >= 20) &
               (wallet_vol['top5_pnl_pct'] <= 0.6) 
               & (wallet_vol['return'] >= 0.1)
               ])

In [ ]:
filtered_wallets = wallet_vol[
    (wallet_vol['num_buckets'] >= 20) &
    (wallet_vol['num_buckets'] <= 300) & 
    (wallet_vol['top5_pnl_pct'] <= 0.4) &
    (wallet_vol['top_market_pnl_pct'] <= 0.5) &
    (wallet_vol['return'] >= 0.1)
].sort_values("pnl_volatility")

print(f"Filtered wallets: {len(filtered_wallets):}")

filtered_wallets.head(20)

In [ ]:
# print deciles of pnl_volatility for filtered wallets

for i in range(0, 11):
    q = i / 10
    vol_q = filtered_wallets["pnl_volatility"].quantile(q)
    print(f"  Volatility at {q:.0%} pct: {vol_q:.4f}")

In [ ]:
wallet_vol[wallet_vol['num_buckets'] >= 2].sort_values("num_buckets").head(10)

In [ ]:
# # display all rows
# pd.set_option('display.max_rows', None)

# buckets[
#     (buckets['wallet'] == WALLET) 
#     # (buckets['dt'] >= pd.Timestamp('2026-01-03T19:00:00', tz='UTC')) &
#     # (buckets['dt'] < pd.Timestamp('2026-01-04', tz='UTC'))
#      ].sort_values("dt_floored")

In [ ]:
import plotly.express as px

# Select top 10 wallets by total_pnl with at least 100 buckets
# top_wallets = (
#     wallet_vol[wallet_vol['num_buckets'] >= 100]
#     .nlargest(10, 'total_pnl')['wallet']
#     .tolist()
# )
top_wallets = filtered_wallets.head(20)['wallet'].tolist()

# Filter buckets for these wallets and compute cumulative PnL over time
cumulative_pnl = (
    buckets[buckets['wallet'].isin(top_wallets)]
    .sort_values(['wallet', 'dt_floored'])
    .copy()
)
cumulative_pnl['cumulative_pnl'] = cumulative_pnl.groupby('wallet')['pnl'].cumsum()

# Plot
fig = px.line(
    cumulative_pnl,
    x='dt_floored',
    y='cumulative_pnl',
    color='wallet',
    title='Cumulative PnL Over Time by Wallet',
    labels={'dt_floored': 'Time', 'cumulative_pnl': 'Cumulative PnL (USDC)', 'wallet': 'Wallet'}
)
fig.show(renderer='browser')

In [ ]:
# plot cummulative pnl over all wallets
cumulative_pnl_all = (
    buckets[buckets['wallet'].isin(top_wallets)].sort_values('dt_floored')
    .copy()
)
cumulative_pnl_all['cumulative_pnl'] = cumulative_pnl_all['pnl'].cumsum()

 # Plot
fig = px.line(
    cumulative_pnl_all,
    x='dt_floored',
    y='cumulative_pnl',
    title='Cumulative PnL Over Time (All Wallets)',
    labels={'dt_floored': 'Time', 'cumulative_pnl': 'Cumulative PnL (USDC)'}
)
fig.show(renderer='browser')

In [ ]:
top_wallet_set = set(filtered_wallets['wallet'])

top_wallet_df=df[df['wallet'].isin(top_wallet_set)] 

print(f"len(top_wallet_df): {len(top_wallet_df):}")

top_wallet_df[['dt', 'title', 'pseudonym', 'condition_id', 'wallet', 'size', 'price', 'pnl']].sort_values('dt').head(10)

In [ ]:
top_wallet_df.groupby("condition_id").agg(
    title = ("title", "first"),
    total_pnl = ("pnl", "sum"),
    num_trades = ("transactionHash", "count"),
    total_notional = ("notional", "sum"),
    wallet_count = ("wallet", "count")
).sort_values("total_pnl", ascending=False).head(30)

In [ ]:
top_wallet_df[top_wallet_df['condition_id'] == "0x74d786eb72f4de44cb39da160a401bcd1a7c390ae03963c1dc13565cc337b9f5"]